# About Time Series Decomposition

### About this notebook

This notebook was used in the 50.039 Deep Learning course at the Singapore University of Technology and Design.

**Author:** Matthieu DE MARI (matthieu_demari@sutd.edu.sg)

**Version:** 1.0 (06/01/2026)

**Requirements:**
- Python 3 (tested on v3.11.4)
- Matplotlib (tested on v3.7.1)
- Numpy (tested on v1.24.3)
- Pandas (tested on v2.3.3)
- Statsmodels (tested on v0.14.6)

### Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

### 1. Preparing a mock time series

We start with the following code, which creates a synthetic (mock) time series of 100 daily data points starting from January 1, 2020. The generated series is constructed as the sum of three components commonly found in real-world time series:

- **Trend component**: Created with np.linspace(10, 30, 100), this simulates a linearly increasing trend from 10 to 30. Mathematically, this corresponds to $T_t = 10 + \frac{20t}{99} $, where $ t \in \{0, 1, ..., 99\} $.
- **Seasonal component**: Constructed as a sine wave using 5*np.sin(...), it introduces periodic behavior with a fixed amplitude of 5. The formula used to mimic seasonality is then $ S_t = 5 \sin( \frac{2\pi t}{10} )$, producing exactly ten seasonal cycles over the 100-day period.
- **Noise component**: Introduced with np.random.normal(scale=1, size=100), it adds Gaussian white noise with standard deviation 1, representing random fluctuations $R_t = \mathcal{N}(0, 1)$.

Our final time series is then defined as $ Y_t = T_t + S_t + R_t $.

In this notebook, we use a simple additive model (but real-life time series can have more complex combinations), and it is useful for illustrating time series decomposition techniques, where the goal is to recover $ T_t $, $ S_t $ and $ R_t $ from $ Y_t $.

In [ ]:
# Initialize seeds and time indexes
np.random.seed(42)
time_index = pd.date_range(start = '2020-01-01', periods = 100, freq = 'D')

# Create components: trend, seasonality, and noise
trend = np.linspace(10, 30, 100)
seasonality = 5*np.sin(np.linspace(0, 20*np.pi, 100))
noise = np.random.normal(scale = 1, size = 100)

# Combine to form the time series
time_series = trend + seasonality + noise
df = pd.DataFrame({'Date': time_index,
                   'Trend': trend,
                   'Seasonality': seasonality,
                   'Noise': noise,
                   'Value': time_series})
df.set_index('Date', inplace = True)

We show below the original time series and its decomposition, based on the formulas used above.

In [ ]:
# Step 2: Plot all components and the final time series
fig, axs = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

df['Trend'].plot(ax = axs[0], color = 'blue', label = 'Trend')
axs[0].set_title('Trend Component')
axs[0].grid(True)

df['Seasonality'].plot(ax = axs[1], color = 'green', label = 'Seasonality')
axs[1].set_title('Seasonality Component')
axs[1].grid(True)

df['Noise'].plot(ax = axs[2], color = 'red', label = 'Noise (Residual)')
axs[2].set_title('Noise Component')
axs[2].grid(True)

df['Value'].plot(ax = axs[3], color = 'black', label = 'Final Time Series')
axs[3].set_title('Final Time Series (Trend + Seasonality + Noise)')
axs[3].grid(True)

plt.tight_layout()
plt.show()

### 2. Using automated tools to perform a time series decomposition

The decomposition algorithm shown below estimates:
- the trend via a moving-average smoothing operation,
- the seasonal component by averaging detrended values across corresponding positions within each seasonal cycle,
- and the residual as whatever remains after removing both trend and seasonality.

As shown in the code below, the automated procedure is able to closely recover the original trend and seasonal structure used to construct the synthetic time series. Small discrepancies are expected and mainly arise from edge effects in the moving-average estimation and from the presence of random noise. In practice, time series decomposition is most often used as an exploratory analysis tool. It allows us to assess whether a dataset exhibits:
- a strong long-term trend,
- a stable and recurring seasonal pattern,
- or predominantly irregular (noise-driven) behavior.

Identifying clear trend and seasonal components is usually a positive signal for downstream modeling, as it suggests that learning algorithms can exploit these regularities rather than attempting to model pure noise.

**Important note:** Whether a time series follows an additive or multiplicative structure is rarely inferred automatically. Instead, it typically relies on domain expertise and an understanding of how the data is generated. In some applications, the observed signal may follow a multiplicative relationship: $ Y_t = T_t \times S_t \times R_t $.

In such cases, both the mathematical formulation and the decomposition procedure must be adapted accordingly (for example, by working in log-space). More complex decomposition models do exist beyond the additive and multiplicative assumptions. However, these lie outside the scope of 50.039 Deep Learning and will not be covered in this course.

In [ ]:
# Step 2: Perform decomposition using the statsmodels library tools
decomposition = seasonal_decompose(df['Value'], model = 'additive', period = 10)

# Plot results
fig, axes = plt.subplots(4, 1, figsize = (10, 8), sharex = True)

df['Value'].plot(ax = axes[0], title = 'Original Time Series', color = 'black')
decomposition.trend.plot(ax = axes[1], title = 'Trend', color = 'blue')
decomposition.seasonal.plot(ax = axes[2], title = 'Seasonality', color = 'green')
decomposition.resid.plot(ax = axes[3], title = 'Residuals', color = 'red')

plt.tight_layout()
plt.show()

### 3. The math behind the seasonal_decompose

#### **Step 1:** Trend Extraction

To estimate the trend component $ T'_t $, seasonal_decompose uses a moving average to smooth out short-term fluctuations. This is typically done by using the arithmetic mean with a window of time with size $ 2k + 1 $ around each time $ t $. The formula is then:

$$ T'_t = \frac{1}{2k+1} \sum_{i = -k}^{k} Y_{t+i} $$

For a multiplicative model, trend estimation is instead performed using the geometric mean (which you might have seen in Probabilities and Statistics class), as:
$$ T'_t = (\prod_{i = -k}^{k} Y_{t+i})^{\frac{1}{2k+1}}$$


The example below shows for a period of 12 months in monthly data:

If period = 10, a centered moving average with a window size of 12 is applied.

This smooths the series and captures long-term trends.

In [ ]:
# Centered moving average with k = 10
k = 10
df["Trend_manual"] = df["Value"].rolling(window = k, center = True).mean()

# Visualization
plt.figure(figsize=(12, 5))
plt.plot(df.index, df["Value"], label = "Observed time series", color = "black", alpha = 0.6)
plt.plot(df.index, df["Trend_manual"], label = "Manual trend (using moving average as above)", color = "blue", linewidth = 2)
plt.plot(df.index, decomposition.trend, label = "Reference trend (using tools)", color = "red", linestyle = "--")
plt.title("Manual Trend Decomposition Using Moving Average and Comparison")
plt.xlabel("Date")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

#### **Step 2:** Seasonal Extraction

The seasonal component $ S_t $ is computed by removing the trend and averaging across seasons.

For additive decomposition, we use the following formula $ D_t = Y_t - T'_t $. Note that after this operation, we simply have $ D_t \approx S_t + R_t $, as only the seasonal component and residual remain (assuming we managed to extract the trend correctly in the previous step).

Then, since seasonality repeats with a fixed period, the seasonal component is calculated by:
$$ S'_t = \frac{1}{N} \sum_{i=1}^N (Y_{t+iP} - T'_{t+iP}) = \frac{1}{N} \sum_{i=1}^N (D_{t+iP}) $$
where:
- P is the periodicity (e.g., 10 for our dataset),
- N is the number of cycles (e.g., 10 for our dataset).
Both numbers can be inferred by visualizing the original time series.

Each time point in the seasonal cycle is averaged to smooth out irregular fluctuations.

The code below shows how it is done.

In [ ]:
# Detrending first
df["D"] = df["Value"] - df["Trend_manual"]

# Seasonal estimation, using average detrended values per seasonal position
df["season_idx"] = np.arange(len(df)) % k
seasonal_means = (df.groupby("season_idx")["D"].mean())

# Optional: enforce zero-mean constraint (identifiability) 
# Can be muted if needed, but needed for perfect match with automated estimation
seasonal_means -= seasonal_means.mean()

# Map back to full time series
df["S_manual"] = df["season_idx"].map(seasonal_means)

# Visualization
plt.figure(figsize=(12, 5))
plt.plot(df.index, df["Value"], label = "Observed time series", color = "black", alpha = 0.6)
plt.plot(df.index, df["S_manual"], label = "Manual seasonality (using formula above)", color = "blue", linewidth = 2)
plt.plot(df.index, decomposition.seasonal, label = "Reference seasonality (using tools)", color = "red", linestyle = "--")
plt.title("Manual Seasonal Decomposition Formula Above and Comparison")
plt.xlabel("Date")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

#### **Step 3:** Residual Component Extraction

Once the estimated trend and seasonality are removed, the remaining component is the estimated residual ($R'_t$). For additive decomposition, this is simply obtained as $ R't = Y_t - T'_t - S'_t $.

The estimated residual component often captures mostly random noise and other irregular fluctuations that are not explained by the trend or seasonality. Because noise is inherently unpredictable, the residual is typically the least accurate component to estimate in a classical decomposition. However, what matters most is not perfect accuracy, but that the magnitude of the residual remains small relative to the trend and/or seasonal components. This indicates that the decomposition can successfully extract the dominant patterns in the data, and that only a small, noisy remainder is left unexplained. This is often a good sign, as it means that there are patterns in the time series, which can later be exploited by our RNN models to formulate predictions about the time series.

### What's next?

We reached the end of this bonus notebook on time series decomposition. Although it is out-of-scope, we would encourage students to study this topic, as it can be used during the initial exploratory data analysis, which precedes any AI model training. It can yield great insights about whether the dataset will be easy or hard to predict.